# 03 — Construir corpus Kaggle (Mundiales históricos)

Combina **dos datasets** de Kaggle para cubrir todos los Mundiales 1930–2022:

- **`abecklas/fifa-world-cup`** — 1930–2014, datos ricos (fase, estadio, asistencia).
- **`martj42/international-football-results-from-1872-to-2017`** — 1872–2024+, complementa **2018 y 2022**.

**Output:** ~100+ docs `.md` en español:
- `mundiales-por-edicion/` — 22 docs (1 por Mundial 1930–2022)
- `equipos-historico/` — ~85 docs (1 por selección con su historial completo)

In [18]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv

ROOT_NB = Path('..').resolve()
load_dotenv(ROOT_NB / '.env')

if not (os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY')):
    raise RuntimeError(
        'Faltan KAGGLE_USERNAME o KAGGLE_KEY en .env.\n'
        'Sigue las instrucciones de la celda anterior.'
    )

try:
    import kagglehub
except ImportError:
    print('Instalando kagglehub...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub

KAGGLE_DIR = ROOT_NB / 'Corpus_Mundial' / 'kaggle'
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

print('1/2 Descargando abecklas/fifa-world-cup (1930-2014, datos ricos)...')
cache1 = Path(kagglehub.dataset_download('abecklas/fifa-world-cup'))
for csv in cache1.glob('*.csv'):
    shutil.copy(csv, KAGGLE_DIR / csv.name)
    print(f'  ✓ {csv.name}')

print('\n2/2 Descargando martj42/international-football-results-from-1872-to-2017 (1872-2024+)...')
cache2 = Path(kagglehub.dataset_download('martj42/international-football-results-from-1872-to-2017'))
for csv in cache2.glob('*.csv'):
    shutil.copy(csv, KAGGLE_DIR / f'martj42_{csv.name}')
    print(f'  ✓ martj42_{csv.name}')

print(f'\nCSVs en {KAGGLE_DIR.relative_to(ROOT_NB)}:')
for c in sorted(KAGGLE_DIR.glob('*.csv')):
    print(f'  - {c.name}  ({c.stat().st_size / 1024:.1f} KB)')

1/2 Descargando abecklas/fifa-world-cup (1930-2014, datos ricos)...
  ✓ WorldCupMatches.csv
  ✓ WorldCupPlayers.csv
  ✓ WorldCups.csv

2/2 Descargando martj42/international-football-results-from-1872-to-2017 (1872-2024+)...


100%|██████████| 1.21M/1.21M [00:00<00:00, 2.91MB/s]

Extracting files...
  ✓ martj42_former_names.csv
  ✓ martj42_goalscorers.csv
  ✓ martj42_results.csv
  ✓ martj42_shootouts.csv

CSVs en Corpus_Mundial\kaggle:
  - martj42_former_names.csv  (1.7 KB)
  - martj42_goalscorers.csv  (3180.4 KB)
  - martj42_results.csv  (3627.1 KB)
  - martj42_shootouts.csv  (28.0 KB)
  - WorldCupMatches.csv  (233.4 KB)
  - WorldCupPlayers.csv  (2100.2 KB)
  - WorldCups.csv  (1.4 KB)


In [19]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import yaml

## 2. Configuración

In [20]:
ROOT = Path('..').resolve()
KAGGLE_DIR = ROOT / 'Corpus_Mundial' / 'kaggle'
OUTPUT_EDICIONES = KAGGLE_DIR / 'mundiales-por-edicion'
OUTPUT_EQUIPOS = KAGGLE_DIR / 'equipos-historico'
OUTPUT_EDICIONES.mkdir(parents=True, exist_ok=True)
OUTPUT_EQUIPOS.mkdir(parents=True, exist_ok=True)

WORLDCUPS_CSV = KAGGLE_DIR / 'WorldCups.csv'
MATCHES_CSV = KAGGLE_DIR / 'WorldCupMatches.csv'
MARTJ42_CSV = KAGGLE_DIR / 'martj42_results.csv'

for p in (WORLDCUPS_CSV, MATCHES_CSV, MARTJ42_CSV):
    assert p.exists(), f'No se encuentra {p}'

print(f'abecklas WorldCups:   {WORLDCUPS_CSV.name}')
print(f'abecklas Matches:     {MATCHES_CSV.name}')
print(f'martj42 results:      {MARTJ42_CSV.name}')

abecklas WorldCups:   WorldCups.csv
abecklas Matches:     WorldCupMatches.csv
martj42 results:      martj42_results.csv


## 3. Cargar y limpiar datasets

In [21]:
df_cups = pd.read_csv(WORLDCUPS_CSV)
df_matches = pd.read_csv(MATCHES_CSV)
df_martj42 = pd.read_csv(MARTJ42_CSV)

print(f'abecklas WorldCups:        {len(df_cups)} filas')
print(f'abecklas WorldCupMatches:  {len(df_matches)} filas')
print(f'martj42 international:     {len(df_martj42)} filas')

# Filtrar martj42 a partidos de Mundial
df_martj42_wc = df_martj42[df_martj42['tournament'] == 'FIFA World Cup'].copy()
df_martj42_wc['date'] = pd.to_datetime(df_martj42_wc['date'])
df_martj42_wc['year'] = df_martj42_wc['date'].dt.year
print(f'\nmartj42 partidos solo de Mundial: {len(df_martj42_wc)} filas')
print(f'Años cubiertos por martj42: {sorted(df_martj42_wc["year"].unique().tolist())}')

abecklas WorldCups:        20 filas
abecklas WorldCupMatches:  4572 filas
martj42 international:     49287 filas

martj42 partidos solo de Mundial: 1036 filas
Años cubiertos por martj42: [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022, 2026]


In [22]:
df_cups.head()

,Year,Country,Winner,Runners-Up,Third,Fourth,GoalsScored,QualifiedTeams,MatchesPlayed,Attendance
0,1930,Uruguay,Uruguay,Argentina,USA,Yugoslavia,70,13,18,590.549
1,1934,Italy,Italy,Czechoslovakia,Germany,Austria,70,16,17,363.000
2,1938,France,Italy,Hungary,Brazil,Sweden,84,15,18,375.700
3,1950,Brazil,Uruguay,Brazil,Sweden,Spain,88,13,22,1.045.246
4,1954,Switzerland,Germany FR,Hungary,Austria,Uruguay,140,16,26,768.607


In [23]:
df_matches.head()

,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions,Attendance,Half-time Home Goals,Half-time Away Goals,Referee,Assistant 1,Assistant 2,RoundID,MatchID,Home Team Initials,Away Team Initials
0,1930.0,13 Jul 1930 - 15:00,Group 1,Pocitos,Montevideo,France,4.0,1.0,Mexico,,4444.0,3.0,0.0,LOMBARDI Domingo (URU),CRISTOPHE Henry (BEL),REGO Gilberto (BRA),201.0,1096.0,FRA,MEX
1,1930.0,13 Jul 1930 - 15:00,Group 4,Parque Central,Montevideo,USA,3.0,0.0,Belgium,,18346.0,2.0,0.0,MACIAS Jose (ARG),MATEUCCI Francisco (URU),WARNKEN Alberto (CHI),201.0,1090.0,USA,BEL
2,1930.0,14 Jul 1930 - 12:45,Group 2,Parque Central,Montevideo,Yugoslavia,2.0,1.0,Brazil,,24059.0,2.0,0.0,TEJADA Anibal (URU),VALLARINO Ricardo (URU),BALWAY Thomas (FRA),201.0,1093.0,YUG,BRA
3,1930.0,14 Jul 1930 - 14:50,Group 3,Pocitos,Montevideo,Romania,3.0,1.0,Peru,,2549.0,1.0,0.0,WARNKEN Alberto (CHI),LANGENUS Jean (BEL),MATEUCCI Francisco (URU),201.0,1098.0,ROU,PER
4,1930.0,15 Jul 1930 - 16:00,Group 1,Parque Central,Montevideo,Argentina,1.0,0.0,France,,23409.0,0.0,0.0,REGO Gilberto (BRA),SAUCEDO Ulises (BOL),RADULESCU Constantin (ROU),201.0,1085.0,ARG,FRA


In [24]:
df_martj42_wc.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year
1486,1930-07-13,Belgium,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True,1930
1487,1930-07-13,France,Mexico,4.0,1.0,FIFA World Cup,Montevideo,Uruguay,True,1930
1488,1930-07-14,Brazil,Yugoslavia,1.0,2.0,FIFA World Cup,Montevideo,Uruguay,True,1930
1489,1930-07-14,Peru,Romania,1.0,3.0,FIFA World Cup,Montevideo,Uruguay,True,1930
1490,1930-07-15,Argentina,France,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,True,1930


In [25]:
# abecklas: filas vacias al final de WorldCupMatches
df_matches = df_matches.dropna(subset=['Year', 'Home Team Name', 'Away Team Name']).copy()
df_matches['Year'] = df_matches['Year'].astype(int)
print(f'abecklas partidos tras limpiar: {len(df_matches)}')

# abecklas no tiene Mundiales 2018 y 2022. Los agregamos manualmente al df_cups.
WC_EXTRA = pd.DataFrame([
    {'Year': 2018, 'Country': 'Russia', 'Winner': 'France', 'Runners-Up': 'Croatia',
     'Third': 'Belgium', 'Fourth': 'England', 'GoalsScored': 169,
     'QualifiedTeams': 32, 'MatchesPlayed': 64, 'Attendance': 3031768},
    {'Year': 2022, 'Country': 'Qatar', 'Winner': 'Argentina', 'Runners-Up': 'France',
     'Third': 'Croatia', 'Fourth': 'Morocco', 'GoalsScored': 172,
     'QualifiedTeams': 32, 'MatchesPlayed': 64, 'Attendance': 3404252},
])

df_cups = pd.concat([df_cups, WC_EXTRA], ignore_index=True)
df_cups = df_cups.sort_values('Year').reset_index(drop=True)
print(f'\ndf_cups tras agregar 2018 y 2022: {len(df_cups)} ediciones')
print(f'Años: {df_cups["Year"].tolist()}')

abecklas partidos tras limpiar: 852

df_cups tras agregar 2018 y 2022: 22 ediciones
Años: [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]


## 4. Diccionarios de traducción

Los datasets están en inglés. Mapeamos a español.

In [26]:
EQUIPOS_TRAD = {
    'Argentina': 'Argentina',
    'Brazil': 'Brasil',
    'Germany': 'Alemania',
    'Germany FR': 'Alemania Federal',
    'East Germany': 'Alemania Oriental',
    'Italy': 'Italia',
    'France': 'Francia',
    'Spain': 'España',
    'England': 'Inglaterra',
    'Portugal': 'Portugal',
    'Netherlands': 'Países Bajos',
    'Belgium': 'Bélgica',
    'Switzerland': 'Suiza',
    'Austria': 'Austria',
    'Sweden': 'Suecia',
    'Denmark': 'Dinamarca',
    'Norway': 'Noruega',
    'Poland': 'Polonia',
    'Romania': 'Rumanía',
    'Bulgaria': 'Bulgaria',
    'Czechoslovakia': 'Checoslovaquia',
    'Czech Republic': 'República Checa',
    'Slovakia': 'Eslovaquia',
    'Hungary': 'Hungría',
    'Soviet Union': 'Unión Soviética',
    'Russia': 'Rusia',
    'Ukraine': 'Ucrania',
    'Serbia': 'Serbia',
    'Serbia and Montenegro': 'Serbia y Montenegro',
    'Yugoslavia': 'Yugoslavia',
    'Croatia': 'Croacia',
    'Slovenia': 'Eslovenia',
    'Bosnia and Herzegovina': 'Bosnia y Herzegovina',
    'Greece': 'Grecia',
    'Turkey': 'Turquía',
    'Iceland': 'Islandia',
    'Republic of Ireland': 'República de Irlanda',
    'Northern Ireland': 'Irlanda del Norte',
    'Scotland': 'Escocia',
    'Wales': 'Gales',
    'Uruguay': 'Uruguay',
    'Chile': 'Chile',
    'Paraguay': 'Paraguay',
    'Peru': 'Perú',
    'Bolivia': 'Bolivia',
    'Colombia': 'Colombia',
    'Ecuador': 'Ecuador',
    'Venezuela': 'Venezuela',
    'USA': 'Estados Unidos',
    'United States': 'Estados Unidos',
    'Canada': 'Canadá',
    'Mexico': 'México',
    'Costa Rica': 'Costa Rica',
    'Honduras': 'Honduras',
    'Panama': 'Panamá',
    'Jamaica': 'Jamaica',
    'El Salvador': 'El Salvador',
    'Haiti': 'Haití',
    'Trinidad and Tobago': 'Trinidad y Tobago',
    'Cuba': 'Cuba',
    'Japan': 'Japón',
    'Korea Republic': 'Corea del Sur',
    'South Korea': 'Corea del Sur',
    'Korea DPR': 'Corea del Norte',
    'North Korea': 'Corea del Norte',
    'China PR': 'China',
    'Iran': 'Irán',
    'IR Iran': 'Irán',
    'Iraq': 'Iraq',
    'Saudi Arabia': 'Arabia Saudita',
    'Kuwait': 'Kuwait',
    'UAE': 'Emiratos Árabes Unidos',
    'United Arab Emirates': 'Emiratos Árabes Unidos',
    'Israel': 'Israel',
    'Qatar': 'Catar',
    'Australia': 'Australia',
    'New Zealand': 'Nueva Zelanda',
    'Morocco': 'Marruecos',
    'Algeria': 'Argelia',
    'Tunisia': 'Túnez',
    'Egypt': 'Egipto',
    'Cameroon': 'Camerún',
    'Senegal': 'Senegal',
    'Nigeria': 'Nigeria',
    'Ghana': 'Ghana',
    "Cote d'Ivoire": 'Costa de Marfil',
    'Ivory Coast': 'Costa de Marfil',
    'South Africa': 'Sudáfrica',
    'Angola': 'Angola',
    'Togo': 'Togo',
    'Zaire': 'Zaire',
    'DR Congo': 'República Democrática del Congo',
    'Dutch East Indies': 'Indias Orientales Neerlandesas',
    'Indonesia': 'Indonesia',
}

FASES_TRAD = {
    'Group 1': 'Grupo 1', 'Group 2': 'Grupo 2', 'Group 3': 'Grupo 3', 'Group 4': 'Grupo 4',
    'Group 5': 'Grupo 5', 'Group 6': 'Grupo 6', 'Group 7': 'Grupo 7', 'Group 8': 'Grupo 8',
    'Group A': 'Grupo A', 'Group B': 'Grupo B', 'Group C': 'Grupo C', 'Group D': 'Grupo D',
    'Group E': 'Grupo E', 'Group F': 'Grupo F', 'Group G': 'Grupo G', 'Group H': 'Grupo H',
    'First round': 'Primera ronda',
    'Second round': 'Segunda ronda',
    'Round of 16': 'Octavos de final',
    'Quarter-finals': 'Cuartos de final',
    'Semi-finals': 'Semifinales',
    'Match for third place': 'Partido por el tercer puesto',
    'Play-off for third place': 'Partido por el tercer puesto',
    'Third place': 'Tercer puesto',
    'Final': 'Final',
    'Preliminary round': 'Ronda preliminar',
}

PAISES_SEDE_TRAD = {
    'Uruguay': 'Uruguay', 'Italy': 'Italia', 'France': 'Francia', 'Brazil': 'Brasil',
    'Switzerland': 'Suiza', 'Sweden': 'Suecia', 'Chile': 'Chile', 'England': 'Inglaterra',
    'Mexico': 'México', 'Germany': 'Alemania', 'Germany FR': 'Alemania Federal',
    'Argentina': 'Argentina', 'Spain': 'España', 'USA': 'Estados Unidos',
    'United States': 'Estados Unidos', 'Korea/Japan': 'Corea del Sur y Japón',
    'South Africa': 'Sudáfrica', 'Russia': 'Rusia', 'Qatar': 'Catar',
    'Japan': 'Japón', 'Korea Republic': 'Corea del Sur',
}


def trad_equipo(name):
    if pd.isna(name): return ''
    s = str(name).strip()
    return EQUIPOS_TRAD.get(s, s)


def trad_fase(name):
    if pd.isna(name): return ''
    s = str(name).strip()
    return FASES_TRAD.get(s, s)


def trad_sede(name):
    if pd.isna(name): return ''
    s = str(name).strip()
    return PAISES_SEDE_TRAD.get(s, s)

In [27]:
# Detectar equipos sin traduccion en AMBOS datasets
equipos_abecklas = set(df_matches['Home Team Name'].dropna()) | set(df_matches['Away Team Name'].dropna())
equipos_martj42 = set(df_martj42_wc['home_team'].dropna()) | set(df_martj42_wc['away_team'].dropna())
equipos_todos = equipos_abecklas | equipos_martj42

sin_trad = sorted(e for e in equipos_todos if e not in EQUIPOS_TRAD)
if sin_trad:
    print(f'⚠ Equipos SIN traduccion ({len(sin_trad)}):')
    for e in sin_trad:
        print(f'  "{e}"')
    print('\nAgregalos a EQUIPOS_TRAD arriba para mejor calidad.')
else:
    print('✓ Todos los equipos tienen traduccion.')

⚠ Equipos SIN traduccion (11):
  "Cape Verde"
  "Curaçao"
  "C�te d'Ivoire"
  "German DR"
  "Jordan"
  "Uzbekistan"
  "rn">Bosnia and Herzegovina"
  "rn">Republic of Ireland"
  "rn">Serbia and Montenegro"
  "rn">Trinidad and Tobago"
  "rn">United Arab Emirates"

Agregalos a EQUIPOS_TRAD arriba para mejor calidad.


## 5. Helpers

In [28]:
MESES_ES = {
    'jan': 'ene', 'feb': 'feb', 'mar': 'mar', 'apr': 'abr',
    'may': 'may', 'jun': 'jun', 'jul': 'jul', 'aug': 'ago',
    'sep': 'sep', 'oct': 'oct', 'nov': 'nov', 'dec': 'dic',
    'Jan': 'ene', 'Feb': 'feb', 'Mar': 'mar', 'Apr': 'abr',
    'May': 'may', 'Jun': 'jun', 'Jul': 'jul', 'Aug': 'ago',
    'Sep': 'sep', 'Oct': 'oct', 'Nov': 'nov', 'Dec': 'dic',
}


def slugify(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode()
    text = re.sub(r'[^\w\s-]', '', text).strip().lower()
    return re.sub(r'[\s_]+', '-', text)


def fmt_fecha_abecklas(s):
    """'13 Jul 1930 - 15:00' -> '13 jul 1930'"""
    if pd.isna(s):
        return ''
    s = str(s).split(' - ')[0].strip()
    for en, es in MESES_ES.items():
        s = s.replace(en, es)
    return s


def fmt_fecha_martj42(dt):
    """datetime -> '15 dic 2022'"""
    if pd.isna(dt):
        return ''
    s = dt.strftime('%d %b %Y').lower()
    for en, es in MESES_ES.items():
        s = s.replace(en, es)
    return s


def fmt_int(v):
    try:
        return f'{int(float(str(v).replace(".", "").replace(",", ""))):,}'
    except (ValueError, TypeError):
        return str(v)


def write_md(path, frontmatter, titulo, body):
    yaml_block = yaml.safe_dump(frontmatter, allow_unicode=True, sort_keys=False)
    content = f'---\n{yaml_block}---\n\n# {titulo}\n\n{body}\n'
    path.write_text(content, encoding='utf-8')

## 6. Normalizar partidos a esquema común

Unificamos partidos de ambos datasets en un solo DataFrame. Para 1930–2014 usamos abecklas (con fase/estadio); para 2018+2022 usamos martj42 (sin fase pero presente).

In [29]:
# Normalizar abecklas (1930-2014)
ab_norm = pd.DataFrame({
    'year': df_matches['Year'].astype(int),
    'fecha_str': df_matches['Datetime'].apply(fmt_fecha_abecklas),
    'fecha_orden': pd.to_datetime(df_matches['Datetime'].astype(str).str.split(' - ').str[0],
                                  format='%d %b %Y', errors='coerce'),
    'fase': df_matches['Stage'].fillna('').apply(trad_fase),
    'home_es': df_matches['Home Team Name'].apply(trad_equipo),
    'away_es': df_matches['Away Team Name'].apply(trad_equipo),
    'home_en': df_matches['Home Team Name'],
    'away_en': df_matches['Away Team Name'],
    'home_goals': df_matches['Home Team Goals'].astype(int),
    'away_goals': df_matches['Away Team Goals'].astype(int),
    'estadio': df_matches['Stadium'].fillna(''),
    'ciudad': df_matches['City'].fillna(''),
    'source': 'abecklas',
})

# Normalizar martj42 (solo 2018 y 2022, lo demas ya esta en abecklas)
m4_filtered = df_martj42_wc[df_martj42_wc['year'].isin([2018, 2022])].copy()
m4_norm = pd.DataFrame({
    'year': m4_filtered['year'].astype(int),
    'fecha_str': m4_filtered['date'].apply(fmt_fecha_martj42),
    'fecha_orden': m4_filtered['date'],
    'fase': '',
    'home_es': m4_filtered['home_team'].apply(trad_equipo),
    'away_es': m4_filtered['away_team'].apply(trad_equipo),
    'home_en': m4_filtered['home_team'],
    'away_en': m4_filtered['away_team'],
    'home_goals': m4_filtered['home_score'].astype(int),
    'away_goals': m4_filtered['away_score'].astype(int),
    'estadio': '',
    'ciudad': m4_filtered['city'].fillna(''),
    'source': 'martj42',
})

all_matches = pd.concat([ab_norm, m4_norm], ignore_index=True)
print(f'Total partidos combinados: {len(all_matches)}')
print(f'  - de abecklas (1930-2014): {(all_matches["source"] == "abecklas").sum()}')
print(f'  - de martj42 (2018+2022):  {(all_matches["source"] == "martj42").sum()}')
print(f'\nAños cubiertos: {sorted(all_matches["year"].unique().tolist())}')

Total partidos combinados: 980
  - de abecklas (1930-2014): 852
  - de martj42 (2018+2022):  128

Años cubiertos: [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]


## 7. Generar docs — uno por edición de Mundial

In [30]:
def build_mundial_doc(row_cup, partidos):
    year = int(row_cup['Year'])
    pais_sede = trad_sede(row_cup.get('Country', ''))
    campeon = trad_equipo(row_cup.get('Winner', ''))
    subcampeon = trad_equipo(row_cup.get('Runners-Up', ''))
    tercero = trad_equipo(row_cup.get('Third', ''))
    cuarto = trad_equipo(row_cup.get('Fourth', ''))
    goles = fmt_int(row_cup.get('GoalsScored', ''))
    equipos = fmt_int(row_cup.get('QualifiedTeams', ''))
    n_partidos = fmt_int(row_cup.get('MatchesPlayed', ''))
    asistencia = fmt_int(row_cup.get('Attendance', ''))

    titulo = f'Copa Mundial de Fútbol de {year}'

    lines = []
    lines.append(
        f'La Copa Mundial de Fútbol de {year} se disputó en {pais_sede}. '
        f'{campeon} se coronó campeón al vencer a {subcampeon} en la final. '
        f'{tercero} terminó en tercer lugar y {cuarto} en cuarto puesto.'
    )
    lines.append('')
    lines.append('## Datos de la edición')
    lines.append('')
    lines.append('| Campo | Valor |')
    lines.append('|---|---|')
    lines.append(f'| Año | {year} |')
    lines.append(f'| País sede | {pais_sede} |')
    lines.append(f'| Campeón | {campeon} |')
    lines.append(f'| Subcampeón | {subcampeon} |')
    lines.append(f'| Tercer lugar | {tercero} |')
    lines.append(f'| Cuarto lugar | {cuarto} |')
    lines.append(f'| Equipos participantes | {equipos} |')
    lines.append(f'| Partidos jugados | {n_partidos} |')
    lines.append(f'| Goles totales | {goles} |')
    lines.append(f'| Asistencia total | {asistencia} |')
    lines.append('')

    if not partidos.empty:
        lines.append('## Partidos')
        partidos = partidos.sort_values('fecha_orden')

        # Si todos tienen fase (abecklas), agrupar por fase. Sino, listar cronologicamente.
        if (partidos['fase'].astype(str).str.len() > 0).all():
            fases_orden = list(dict.fromkeys(partidos['fase'].tolist()))
            for fase in fases_orden:
                sub = partidos[partidos['fase'] == fase]
                lines.append('')
                lines.append(f'### {fase}')
                lines.append('')
                for _, p in sub.iterrows():
                    lugar = (
                        f'{p["estadio"]}, {p["ciudad"]}'
                        if p['estadio'] and p['ciudad']
                        else p['estadio'] or p['ciudad']
                    )
                    sufijo = f' ({lugar})' if lugar else ''
                    lines.append(
                        f'- {p["fecha_str"]}: {p["home_es"]} {p["home_goals"]}-{p["away_goals"]} {p["away_es"]}{sufijo}'
                    )
        else:
            lines.append('')
            for _, p in partidos.iterrows():
                lugar = p['ciudad']
                sufijo = f' ({lugar})' if lugar else ''
                lines.append(
                    f'- {p["fecha_str"]}: {p["home_es"]} {p["home_goals"]}-{p["away_goals"]} {p["away_es"]}{sufijo}'
                )

    body = '\n'.join(lines)
    return titulo, body


escritos_ediciones = []
for _, row in df_cups.iterrows():
    year = int(row['Year'])
    partidos_year = all_matches[all_matches['year'] == year]
    titulo, body = build_mundial_doc(row, partidos_year)

    fm = {
        'titulo': titulo,
        'tema': 'historia-mundial',
        'tipo': 'historico',
        'fuente': 'Kaggle: abecklas/fifa-world-cup + martj42/international-football-results',
        'edicion': year,
        'tags': ['mundial', 'historia', str(year)],
    }
    path = OUTPUT_EDICIONES / f'mundial-{year}.md'
    write_md(path, fm, titulo, body)
    escritos_ediciones.append(path)
    print(f'  ✓ {path.name}  ({len(body):,} chars)')

print(f'\nTotal ediciones escritas: {len(escritos_ediciones)}')

  ✓ mundial-1930.md  (1,823 chars)
  ✓ mundial-1934.md  (1,666 chars)
  ✓ mundial-1938.md  (1,781 chars)
  ✓ mundial-1950.md  (2,200 chars)
  ✓ mundial-1954.md  (2,244 chars)
  ✓ mundial-1958.md  (2,954 chars)
  ✓ mundial-1962.md  (2,944 chars)
  ✓ mundial-1966.md  (2,878 chars)
  ✓ mundial-1970.md  (2,764 chars)
  ✓ mundial-1974.md  (3,219 chars)
  ✓ mundial-1978.md  (3,840 chars)
  ✓ mundial-1982.md  (4,049 chars)
  ✓ mundial-1986.md  (4,319 chars)
  ✓ mundial-1990.md  (4,224 chars)
  ✓ mundial-1994.md  (4,180 chars)
  ✓ mundial-1998.md  (5,023 chars)
  ✓ mundial-2002.md  (5,306 chars)
  ✓ mundial-2006.md  (5,765 chars)
  ✓ mundial-2010.md  (5,663 chars)
  ✓ mundial-2014.md  (6,269 chars)
  ✓ mundial-2018.md  (3,555 chars)
  ✓ mundial-2022.md  (3,490 chars)

Total ediciones escritas: 22


## 8. Generar docs — uno por equipo histórico

In [31]:
def build_team_doc(team_es, partidos):
    """Construye doc para un equipo con todo su historial en Mundiales."""
    partidos = partidos.sort_values('fecha_orden')

    # Stats
    victorias = empates = derrotas = 0
    gf = gc = 0
    for _, p in partidos.iterrows():
        es_local = (p['home_es'] == team_es)
        if es_local:
            gp, gr = int(p['home_goals']), int(p['away_goals'])
        else:
            gp, gr = int(p['away_goals']), int(p['home_goals'])
        gf += gp; gc += gr
        if gp > gr:    victorias += 1
        elif gp == gr: empates += 1
        else:          derrotas += 1

    ediciones = sorted(partidos['year'].unique().tolist())
    n_partidos = len(partidos)
    n_ediciones = len(ediciones)

    titulo = f'{team_es} en Copas del Mundo'
    lines = []
    lines.append(
        f'{team_es} ha participado en {n_ediciones} ediciones de la Copa Mundial de Fútbol, '
        f'disputando un total de {n_partidos} partidos.'
    )
    lines.append('')
    lines.append('## Estadísticas globales')
    lines.append('')
    lines.append('| Indicador | Valor |')
    lines.append('|---|---:|')
    lines.append(f'| Ediciones participadas | {n_ediciones} |')
    lines.append(f'| Partidos jugados | {n_partidos} |')
    lines.append(f'| Victorias | {victorias} |')
    lines.append(f'| Empates | {empates} |')
    lines.append(f'| Derrotas | {derrotas} |')
    lines.append(f'| Goles a favor | {gf} |')
    lines.append(f'| Goles en contra | {gc} |')
    lines.append(f'| Diferencia de goles | {gf - gc:+d} |')
    lines.append('')
    lines.append('## Participación por edición')
    lines.append('')

    for year in ediciones:
        sub = partidos[partidos['year'] == year]
        lines.append(f'### {int(year)}')
        lines.append('')
        for _, p in sub.iterrows():
            fase = f' ({p["fase"]})' if p['fase'] else ''
            lines.append(
                f'- {p["fecha_str"]}: {p["home_es"]} {p["home_goals"]}-{p["away_goals"]} {p["away_es"]}{fase}'
            )
        lines.append('')

    return titulo, '\n'.join(lines)


# Conjunto de equipos en español (de ambos datasets)
teams_es = sorted(set(all_matches['home_es']) | set(all_matches['away_es']))
teams_es = [t for t in teams_es if t]  # quitar vacios
print(f'Equipos únicos a procesar: {len(teams_es)}')

escritos_equipos = []
for team_es in teams_es:
    sub = all_matches[(all_matches['home_es'] == team_es) | (all_matches['away_es'] == team_es)]
    if sub.empty:
        continue
    titulo, body = build_team_doc(team_es, sub)

    fm = {
        'titulo': titulo,
        'tema': 'equipo-historico',
        'tipo': 'agregado',
        'fuente': 'Kaggle: abecklas/fifa-world-cup + martj42/international-football-results',
        'equipo': team_es,
        'tags': ['equipo', 'historico', slugify(team_es)],
    }
    path = OUTPUT_EQUIPOS / f'equipo-{slugify(team_es)}.md'
    write_md(path, fm, titulo, body)
    escritos_equipos.append(path)

print(f'Total equipos escritos: {len(escritos_equipos)}')

Equipos únicos a procesar: 85
Total equipos escritos: 85


## 9. Resumen final

In [32]:
print('=' * 60)
print('RESUMEN — Corpus Kaggle')
print('=' * 60)
n_ediciones = len(list(OUTPUT_EDICIONES.glob('*.md')))
n_equipos = len(list(OUTPUT_EQUIPOS.glob('*.md')))
print(f'Ediciones de Mundial:  {n_ediciones:3d} docs')
print(f'Equipos históricos:    {n_equipos:3d} docs')
print(f'TOTAL:                 {n_ediciones + n_equipos:3d} docs')

RESUMEN — Corpus Kaggle
Ediciones de Mundial:   22 docs
Equipos históricos:     85 docs
TOTAL:                 107 docs


In [33]:
# Preview Mundial 2022 (la nueva edicion)
p = OUTPUT_EDICIONES / 'mundial-2022.md'
if p.exists():
    print(f'--- Preview: {p.name} ---')
    print(p.read_text(encoding='utf-8')[:2000])

--- Preview: mundial-2022.md ---
---
titulo: Copa Mundial de Fútbol de 2022
tema: historia-mundial
tipo: historico
fuente: 'Kaggle: abecklas/fifa-world-cup + martj42/international-football-results'
edicion: 2022
tags:
- mundial
- historia
- '2022'
---

# Copa Mundial de Fútbol de 2022

La Copa Mundial de Fútbol de 2022 se disputó en Catar. Argentina se coronó campeón al vencer a Francia en la final. Croacia terminó en tercer lugar y Marruecos en cuarto puesto.

## Datos de la edición

| Campo | Valor |
|---|---|
| Año | 2022 |
| País sede | Catar |
| Campeón | Argentina |
| Subcampeón | Francia |
| Tercer lugar | Croacia |
| Cuarto lugar | Marruecos |
| Equipos participantes | 32 |
| Partidos jugados | 64 |
| Goles totales | 172 |
| Asistencia total | 3,404,252 |

## Partidos

- 20 nov 2022: Catar 0-2 Ecuador (Al Khor)
- 21 nov 2022: Senegal 0-2 Países Bajos (Doha)
- 21 nov 2022: Inglaterra 6-2 Irán (Al Rayyan)
- 21 nov 2022: Estados Unidos 1-1 Gales (Al Rayyan)
- 22 nov 2022: Argentin

In [34]:
# Preview equipo Argentina (campeon 2022)
p = OUTPUT_EQUIPOS / 'equipo-argentina.md'
if p.exists():
    print(f'--- Preview: {p.name} ---')
    print(p.read_text(encoding='utf-8')[:2500])

--- Preview: equipo-argentina.md ---
---
titulo: Argentina en Copas del Mundo
tema: equipo-historico
tipo: agregado
fuente: 'Kaggle: abecklas/fifa-world-cup + martj42/international-football-results'
equipo: Argentina
tags:
- equipo
- historico
- argentina
---

# Argentina en Copas del Mundo

Argentina ha participado en 18 ediciones de la Copa Mundial de Fútbol, disputando un total de 92 partidos.

## Estadísticas globales

| Indicador | Valor |
|---|---:|
| Ediciones participadas | 18 |
| Partidos jugados | 92 |
| Victorias | 49 |
| Empates | 18 |
| Derrotas | 25 |
| Goles a favor | 154 |
| Goles en contra | 102 |
| Diferencia de goles | +52 |

## Participación por edición

### 1930

- 15 jul 1930: Argentina 1-0 Francia (Grupo 1)
- 19 jul 1930: Argentina 6-3 México (Grupo 1)
- 22 jul 1930: Argentina 3-1 Chile (Grupo 1)
- 26 jul 1930: Argentina 6-1 Estados Unidos (Semifinales)
- 30 jul 1930: Uruguay 4-2 Argentina (Final)

### 1934

- 27 may 1934: Suecia 3-2 Argentina (Ronda preliminar)
